In [2]:
%pip install chromadb

In [3]:
documents = [
    "To resolve intermittent packet loss on fiber lines, first check the ONT signal strength via the optical power meter. Readings below -28dBm indicate a degraded fiber connection requiring a technician dispatch.",
    "Router reboot loops are commonly caused by firmware corruption or power supply instability. Recommended steps: factory reset, then reflash firmware; if the loop persists after reflash, escalate to hardware replacement.",
    "Billing disputes over sudden charge increases should first be checked against the customer's plan change history -- most spikes trace back to an expired promotional rate rather than a billing error.",
    "Service outages affecting multiple addresses in the same area typically indicate a shared infrastructure failure (e.g. a damaged distribution cable or a failed aggregation switch), not individual customer equipment.",
    "New installation appointments that are missed or delayed should be re-scheduled within 48 hours per SLA; repeated missed appointments should be escalated to the field operations supervisor.",
    "Wrong equipment shipped for an installation (e.g. incompatible ONT model) should be flagged to the provisioning team immediately, since it blocks activation and risks a second missed-appointment SLA breach.",
    "Evening-only speed degradation (peak-hour slowdown) is usually a capacity/congestion issue on the shared local node, not a fault with the customer's individual line -- check node utilization graphs before dispatching a technician.",
    "DDoS-related traffic spikes on the network core should be distinguished from legitimate demand surges by checking source IP diversity -- thousands of unique sources hitting one endpoint is the key signature.",
]

metadata = [
    {"topic": "signal_quality"},
    {"topic": "hardware_fault"},
    {"topic": "billing"},
    {"topic": "service_outage"},
    {"topic": "provisioning"},
    {"topic": "provisioning"},
    {"topic": "performance_degradation"},
    {"topic": "security"},
]

In [4]:
import chromadb
from chromadb.utils import embedding_functions

# Use the same embedding model from Day 17, wired into Chroma's interface
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.Client()  # in-memory for now; PersistentClient(path=...) would save to disk
collection = client.create_collection(name="telecom_ops_kb", embedding_function=embedding_fn)

collection.add(
    documents=documents,
    metadatas=metadata,
    ids=[f"doc_{i}" for i in range(len(documents))],
)

print(f"Added {collection.count()} documents to the index.")

Added 8 documents to the index.


In [5]:
def search(query, n_results=3):
    results = collection.query(query_texts=[query], n_results=n_results)
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        # Chroma returns distance (lower = more similar) by default, not similarity -- worth noting explicitly
        print(f"distance={dist:.3f}  topic={meta['topic']}")
        print(f"  {doc}")
    print()

search("customer's internet is really slow at night")
search("technician didn't show up for scheduled install")
search("why did my monthly cost go up")

distance=0.408  topic=performance_degradation
  Evening-only speed degradation (peak-hour slowdown) is usually a capacity/congestion issue on the shared local node, not a fault with the customer's individual line -- check node utilization graphs before dispatching a technician.
distance=0.619  topic=signal_quality
  To resolve intermittent packet loss on fiber lines, first check the ONT signal strength via the optical power meter. Readings below -28dBm indicate a degraded fiber connection requiring a technician dispatch.
distance=0.720  topic=security
  DDoS-related traffic spikes on the network core should be distinguished from legitimate demand surges by checking source IP diversity -- thousands of unique sources hitting one endpoint is the key signature.

distance=0.548  topic=provisioning
  Wrong equipment shipped for an installation (e.g. incompatible ONT model) should be flagged to the provisioning team immediately, since it blocks activation and risks a second missed-appointment

In [6]:
search("what's a good recipe for pasta")  # should come back with clearly worse (higher) distances than any real query above

distance=0.983  topic=hardware_fault
  Router reboot loops are commonly caused by firmware corruption or power supply instability. Recommended steps: factory reset, then reflash firmware; if the loop persists after reflash, escalate to hardware replacement.
distance=0.996  topic=security
  DDoS-related traffic spikes on the network core should be distinguished from legitimate demand surges by checking source IP diversity -- thousands of unique sources hitting one endpoint is the key signature.
distance=1.014  topic=performance_degradation
  Evening-only speed degradation (peak-hour slowdown) is usually a capacity/congestion issue on the shared local node, not a fault with the customer's individual line -- check node utilization graphs before dispatching a technician.



In [7]:
import requests

def generate_answer(question, context_chunks):
    context = "\n\n".join([f"<doc>{chunk}</doc>" for chunk in context_chunks])
    
    prompt = f"""You are a telecom network operations assistant. Answer the operator's question using ONLY the information in the documents below. If the documents don't contain enough information to answer, say so explicitly -- do not make up an answer.

<documents>
{context}
</documents>

<question>{question}</question>

Answer concisely, based only on the documents above:"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:3b",
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0},
        },
    )
    return response.json()["response"].strip()

In [8]:
DISTANCE_THRESHOLD = 0.7  # based on Day 18's finding: real matches ~0.4-0.6, off-topic ~0.98+

def rag_answer(question, n_results=3):
    results = collection.query(query_texts=[question], n_results=n_results)
    docs = results["documents"][0]
    distances = results["distances"][0]
    
    # Only keep chunks that actually pass the relevance bar
    relevant_docs = [d for d, dist in zip(docs, distances) if dist < DISTANCE_THRESHOLD]
    
    print(f"Retrieved {len(docs)} chunks, {len(relevant_docs)} passed threshold (<{DISTANCE_THRESHOLD})")
    for d, dist in zip(docs, distances):
        print(f"  [{'KEPT' if dist < DISTANCE_THRESHOLD else 'dropped'}] dist={dist:.3f}")
    
    if not relevant_docs:
        return "No relevant information found in the knowledge base for this question."
    
    answer = generate_answer(question, relevant_docs)
    return answer

In [9]:
questions = [
    "Why is the customer's internet slow every evening?",
    "A customer's install technician never showed up, what should I do?",
    "What's the recommended pizza topping?",  # should trigger the "no relevant info" path
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("=" * 60)

Q: Why is the customer's internet slow every evening?
Retrieved 3 chunks, 3 passed threshold (<0.7)
  [KEPT] dist=0.404
  [KEPT] dist=0.667
  [KEPT] dist=0.692
A: The customer's internet slow every evening is likely due to a capacity/congestion issue on the shared local node, as indicated by the node utilization graphs.
Q: A customer's install technician never showed up, what should I do?
Retrieved 3 chunks, 2 passed threshold (<0.7)
  [KEPT] dist=0.532
  [KEPT] dist=0.674
  [dropped] dist=0.795
A: According to the documents, since the technician did not show up, the appointment should be re-scheduled within 48 hours per SLA.
Q: What's the recommended pizza topping?
Retrieved 3 chunks, 0 passed threshold (<0.7)
  [dropped] dist=0.930
  [dropped] dist=0.990
  [dropped] dist=1.013
A: No relevant information found in the knowledge base for this question.


In [10]:
print(rag_answer("What's the capital of France?"))

Retrieved 3 chunks, 0 passed threshold (<0.7)
  [dropped] dist=1.016
  [dropped] dist=1.038
  [dropped] dist=1.039
No relevant information found in the knowledge base for this question.


In [11]:
eval_set = [
    {"question": "customer's speeds tank every night around dinner time", "expected_topic": "performance_degradation"},
    {"question": "fiber line has been dropping packets on and off", "expected_topic": "signal_quality"},
    {"question": "router won't stop restarting itself", "expected_topic": "hardware_fault"},
    {"question": "why was I billed more than usual this cycle", "expected_topic": "billing"},
    {"question": "whole street lost service at the same time", "expected_topic": "service_outage"},
    {"question": "install appointment got missed again", "expected_topic": "provisioning"},
    {"question": "wrong ONT model delivered for new setup", "expected_topic": "provisioning"},
    {"question": "seeing a flood of traffic from thousands of different IPs", "expected_topic": "security"},
    {"question": "what's the best pizza topping", "expected_topic": None},   # should retrieve nothing
    {"question": "who is the president of the united states", "expected_topic": None},  # should retrieve nothing
]

In [12]:
def evaluate_retrieval(eval_set, threshold=0.7):
    correct = 0
    results_log = []
    
    for item in eval_set:
        q, expected = item["question"], item["expected_topic"]
        results = collection.query(query_texts=[q], n_results=1)
        top_dist = results["distances"][0][0]
        top_topic = results["metadatas"][0][0]["topic"]
        
        retrieved_topic = top_topic if top_dist < threshold else None
        is_correct = (retrieved_topic == expected)
        correct += is_correct
        
        results_log.append({
            "question": q, "expected": expected, "retrieved": retrieved_topic,
            "distance": round(top_dist, 3), "correct": is_correct
        })
    
    accuracy = correct / len(eval_set)
    print(f"Retrieval accuracy: {correct}/{len(eval_set)} = {accuracy:.1%}\n")
    for r in results_log:
        status = "✓" if r["correct"] else "✗"
        print(f"{status} '{r['question']}' -> expected={r['expected']}, got={r['retrieved']} (dist={r['distance']})")
    return results_log

retrieval_results = evaluate_retrieval(eval_set)

Retrieval accuracy: 10/10 = 100.0%

✓ 'customer's speeds tank every night around dinner time' -> expected=performance_degradation, got=performance_degradation (dist=0.589)
✓ 'fiber line has been dropping packets on and off' -> expected=signal_quality, got=signal_quality (dist=0.29)
✓ 'router won't stop restarting itself' -> expected=hardware_fault, got=hardware_fault (dist=0.292)
✓ 'why was I billed more than usual this cycle' -> expected=billing, got=billing (dist=0.458)
✓ 'whole street lost service at the same time' -> expected=service_outage, got=service_outage (dist=0.466)
✓ 'install appointment got missed again' -> expected=provisioning, got=provisioning (dist=0.409)
✓ 'wrong ONT model delivered for new setup' -> expected=provisioning, got=provisioning (dist=0.473)
✓ 'seeing a flood of traffic from thousands of different IPs' -> expected=security, got=security (dist=0.373)
✓ 'what's the best pizza topping' -> expected=None, got=None (dist=0.932)
✓ 'who is the president of the unit

In [13]:
def faithfulness_check(eval_set, retrieval_results):
    for item, r in zip(eval_set, retrieval_results):
        if r["expected"] is None or not r["correct"]:
            continue  # only check faithfulness where retrieval succeeded
        answer = rag_answer(item["question"])
        print(f"Q: {item['question']}")
        print(f"A: {answer}\n")

faithfulness_check(eval_set, retrieval_results)

Retrieved 3 chunks, 1 passed threshold (<0.7)
  [KEPT] dist=0.589
  [dropped] dist=0.798
  [dropped] dist=0.878
Q: customer's speeds tank every night around dinner time
A: The issue may be related to capacity/congestion on the shared local node, rather than a fault with the customer's individual line. Checking node utilization graphs before dispatching a technician may be helpful.

Retrieved 3 chunks, 2 passed threshold (<0.7)
  [KEPT] dist=0.290
  [KEPT] dist=0.674
  [dropped] dist=0.702
Q: fiber line has been dropping packets on and off
A: Based on the documents, it's unclear what's causing the intermittent packet loss on the fiber line. The documents don't provide enough information to determine if it's a problem with the fiber connection, the ONT signal strength, or something else.

Retrieved 3 chunks, 1 passed threshold (<0.7)
  [KEPT] dist=0.292
  [dropped] dist=0.708
  [dropped] dist=0.802
Q: router won't stop restarting itself
A: The router won't stop restarting itself due to a